In [0]:
%pip install transformers accelerate chromadb

In [0]:
dbutils.library.restartPython()

In [0]:
%pip install --upgrade opentelemetry-api opentelemetry-sdk

In [0]:
from transformers import pipeline
import chromadb

from chromadb.config import Settings

In [0]:
VECTOR_DB_PATH = "/tmp/vector_db-test/"

client = chromadb.PersistentClient(path=VECTOR_DB_PATH)

collection = client.get_or_create_collection("legal_knowledge")

In [0]:
qa_model = pipeline(
    "text-generation",
    model="google/flan-t5-large",
    max_length=512
)

In [0]:
def retrieve_chunks(query, k=5):
    # Check if collection has documents before querying
    if collection.count() == 0:
        return [], []
    
    results = collection.query(
        query_texts=[query],
        n_results=k
    )
    return results["documents"][0], results["metadatas"][0]

In [0]:
def refine_context(docs):
    seen = set()
    refined = []

    for doc in docs:
        doc = doc.strip()

        if len(doc) < 100:
            continue

        if doc not in seen:
            refined.append(doc)
            seen.add(doc)

    return refined

In [0]:
def build_prompt(query, context):
    context_text = "\n\n".join(context[:3])

    prompt = f"""
You are a legal assistant helping Indian citizens understand laws.

Using the legal context below, answer the question clearly.

Explain in simple language.
Mention legal section numbers if available.
Include penalties if applicable.
Provide practical guidance.

Question:
{query}

Legal Context:
{context_text}

Answer:
"""
    return prompt

In [0]:
def generate_answer(query):
    docs, metadata = retrieve_chunks(query)
    context = refine_context(docs)

    prompt = build_prompt(query, context)

    response = qa_model(prompt)[0]["generated_text"]

    return response, metadata

In [0]:
def format_output(answer, metadata):
    sections = {m.get("section","") for m in metadata if m.get("section")}

    formatted = f"""
⚖️ Legal Explanation:

{answer}

📚 Relevant Sections:
{", ".join(sections) if sections else "Refer to applicable legal provisions"}

⚠️ Disclaimer:
This response is AI-generated legal information and not a substitute for professional legal advice.
"""

    return formatted

In [0]:
query = "What is the penalty for not wearing a helmet?"

answer, metadata = generate_answer(query)

print(format_output(answer, metadata))